# v2: is the distillation gain the teacher, or regularisation? (one seed per run)

Six conditions on identical data, LoRA config and optimizer, differing only in what is listed:

| condition | what differs from `sft_only` |
|---|---|
| `base` | no training |
| `sft_only` | (reference) hard-label SFT, 3 epochs |
| `sft_early` | stopped after 1 epoch (early-stopping control) |
| `sft_ls` | label smoothing 0.1 (regularisation control) |
| `self_distill` | 0.5 SFT + 0.5 KL, but the teacher is the frozen base student itself: no external knowledge |
| `distilled` | 0.5 SFT + 0.5 KL from Qwen3-14B |
| `distilled_8b` (optional) | same, from Qwen3-8B |

If `self_distill` matches `distilled`, the gain is regularisation; if `distilled` beats it, the teacher contributes information.

**How to run.** Set `SEED` in the first code cell (0, then 1, then 2), Save Version -> Save & Run All. Each seed has its own run tag (`v2-seed<SEED>`) and resumes on its own. Needs the Kaggle secret `HF_TOKEN` (write access to a Hugging Face repo); never paste a token into a cell. Enable GPU T4 x2 and Internet.

Rough time per seed on T4 x2: training ~70 min, evaluation ~3.5 h (221 tasks per condition: 121 synthetic multi-step tasks on unseen tools + 100 held-out single-step tasks on the training tools), so ~5 h. The base model is evaluated only for seed 0 (greedy decoding makes it identical across seeds).

In [1]:
# ---- edit these two lines, then Save & Run All. One seed per run: 0, then 1, then 2. ----
SEED = 0
INCLUDE_8B = False    # also train/evaluate the Qwen3-8B-teacher condition (~+25 min per seed)

import base64
import os
import subprocess
import sys

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['ADBENCH_SEED'] = str(SEED)

# Kaggle secrets (Add-ons -> Secrets), all optional:
#   GH_TOKEN  read access to the GitHub repo, needed only while the repo is private
#   HF_TOKEN  write access to a Hugging Face repo, needed only to resume across sessions
os.environ.setdefault('ADBENCH_HF_REPO', 'NahlaNabil/adbench-run')
os.environ['ADBENCH_RUN_TAG'] = f'v2-seed{SEED}'   # one run tag per seed
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    for _name in ('GH_TOKEN', 'HF_TOKEN'):
        try:
            os.environ[_name] = _secrets.get_secret(_name)
        except Exception:
            pass
except Exception:
    pass


def git(*args, timeout=600):
    """Run git without ever prompting (a credentials prompt would hang an unattended run for
    hours). Uses GH_TOKEN when set; if that fails (revoked token, or a public repo that needs
    none) it retries once without it."""
    env = {**os.environ, 'GIT_TERMINAL_PROMPT': '0'}
    token = os.environ.get('GH_TOKEN')
    for use_token in ([True, False] if token else [False]):
        cmd = ['git']
        if use_token:
            basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
            cmd += ['-c', f'http.https://github.com/.extraheader=AUTHORIZATION: basic {basic}']
        try:
            subprocess.run(cmd + list(args), check=True, timeout=timeout, env=env)
            return
        except subprocess.CalledProcessError:
            if not use_token:
                raise
            print('git with GH_TOKEN failed; retrying without it.')


def run_module(*args, timeout=4 * 3600):
    """Run `python -m <args>` in a fresh process, print the tail of its output, and raise if it
    fails or exceeds `timeout` seconds (a bare `!` command never stops the notebook)."""
    proc = subprocess.run(
        [sys.executable, '-m', *args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, timeout=timeout
    )
    print(proc.stdout[-20000:])
    proc.check_returncode()


ON_KAGGLE = os.path.isdir('/kaggle')
REPO_DIR = '/kaggle/working/agentic-distillation-benchmark' if ON_KAGGLE else '/content/agentic-distillation-benchmark'

if not os.path.isdir(REPO_DIR):
    git('clone', 'https://github.com/Nahla-Nabil/agentic-distillation-benchmark.git', REPO_DIR)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True, timeout=1800)

src_path = os.path.join(REPO_DIR, 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
os.environ['PYTHONPATH'] = src_path + os.pathsep + os.environ.get('PYTHONPATH', '')

Cloning into '/kaggle/working/agentic-distillation-benchmark'...
fatal: could not read Username for 'https://github.com': terminal prompts disabled
Cloning into '/kaggle/working/agentic-distillation-benchmark'...


git with GH_TOKEN failed; retrying without it.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.2/81.2 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 80.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 112.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 91.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 91.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 10.8 MB/s eta 0:00:00
  

In [2]:
# re-sync to the latest commit
git('-C', REPO_DIR, 'checkout', '--', '.')
git('-C', REPO_DIR, 'pull')

fatal: could not read Username for 'https://github.com': terminal prompts disabled


git with GH_TOKEN failed; retrying without it.
Already up to date.


In [3]:
from adbench import pipeline_state as ps

ps.restore()
ps.check_upload()

Fetching 0 files: 0it [00:00, ?it/s]

[persist] nothing to restore from NahlaNabil/adbench-run (tag 'v2-seed0') — starting fresh.
[persist] ON — write access to NahlaNabil/adbench-run confirmed (tag 'v2-seed0').


In [4]:
CONDITIONS_V2 = ["base", "sft_only", "sft_early", "sft_ls", "self_distill", "distilled"] + (["distilled_8b"] if INCLUDE_8B else [])
TRAINED = [c for c in CONDITIONS_V2 if c != "base"]
EVAL_CONDITIONS = [c for c in CONDITIONS_V2 if c != "base" or SEED == 0]
print("seed", SEED, "| train:", CONDITIONS_V2, "| evaluate:", EVAL_CONDITIONS)

seed 0 | train: ['base', 'sft_only', 'sft_early', 'sft_ls', 'self_distill', 'distilled'] | evaluate: ['base', 'sft_only', 'sft_early', 'sft_ls', 'self_distill', 'distilled']


## 1. Data

In [5]:
def prepare_data():
    run_module("adbench.data.prepare", "--config", "configs/data.yaml")
    run_module("adbench.data.general_eval", "--config", "configs/experiment.yaml")


ps.run_stage("data", prepare_data)

[stage] data: running ...
Loading glaiveai/glaive-function-calling-v2 ...

Generating train split: 100%|██████████| 112960/112960 [00:02<00:00, 50842.07 examples/s]
Loaded 112960 raw rows.
  ...0/112960
  ...20000/112960
  ...40000/112960
  ...60000/112960
  ...80000/112960
  ...100000/112960

=== prepare.py summary ===
raw rows processed: 112960
drop reasons: {'zero_calls': 49742, 'multi_call': 19583, 'tool_not_selected': 29073, 'ok': 10126, 'non_canonical_args': 3437, 'parse_error': 999}
distinct tool names seen in raw data: 966
kept examples per selected tool (before the per-tool cap):
  calculate_age: 1722
  calculate_bmi: 2946
  calculate_discount: 1852
  calculate_distance: 189
  calculate_tip: 1913
  convert_currency: 1081
  generate_random_number: 240
  get_stock_price: 183
train: 640   test: 160   total: 800
wrote: /kaggle/working/agentic-distillation-benchmark/data/splits/train.jsonl
wrote: /kaggle/working/agentic-distillation-benchmark/data/splits/test.jsonl
wrote: /kaggle/w

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

[persist] saved to NahlaNabil/adbench-run (stage data)


True

## 2. Training

In [6]:
for condition in CONDITIONS_V2:
    ps.run_stage(
        f"train_{condition}",
        lambda condition=condition: run_module("adbench.training.train", "--condition", condition),
    )

[stage] train_base: running ...
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.7: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!

Loading weights: 100%|██████████| 398/398 [00:01<00:00, 220.48it/s]
Unsloth 2026.9.7 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/agentic-distillation-benchmark/checkpoints/base/tokenizer_config.json.
{
  "condition": "base",
  "checkpoint_dir": "/kaggle/working/agentic-distillation-benc

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


[persist] saved to NahlaNabil/adbench-run (stage train_distilled)


Guard: stop now if a trained model does not produce clean tool calls, before the long evaluation.

In [7]:
def guard():
    for condition in TRAINED:
        run_module("adbench.evaluation.diagnose", "--condition", condition,
                   "--skip-training-target", "--n", "5", "--require-tool-call", "--max-failures", "1")


ps.run_stage("guard", guard)

[stage] guard: running ...
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.7: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
total weights      : 3.257 GiB
no_split classes   : ['Qwen3DecoderLayer']
output head        : lm_head -> cuda:1
head headroom      : 0.359 GiB
activation reserve : 11.166 GiB requested
tied to head       : ['model.embed_tokens']
  cuda:0  budget  12.95 GiB  weights  1.760 GiB  free 11.192 GiB  reserve 11.166 GiB
  cuda:1  budget  13.00 GiB  weights  1.497 GiB  free 11.499 GiB  rese

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


[persist] saved to NahlaNabil/adbench-run (stage guard)


True

## 3. Evaluation (greedy decoding; unseen-tool and seen-tool sets)

In [8]:
import json

from adbench.evaluation.run_eval import evaluate_condition, write_eval_outputs
from adbench.training.train import REPO_ROOT, load_experiment_config

experiment_config = load_experiment_config("configs/experiment.yaml")
models_config = load_experiment_config("configs/models.yaml")
chain_lengths = experiment_config["harness"]["chain_lengths"]

all_rows = []
perplexities = {}
for condition in EVAL_CONDITIONS:
    stage = f"eval_{condition}"
    cache_file = ps.stages_dir() / f"{stage}.json"
    if ps.is_done(stage) and cache_file.exists():
        saved = json.loads(cache_file.read_text(encoding="utf-8"))
        rows, perplexity = saved["rows"], saved["perplexity"]
        print(f"=== {condition}: restored from an earlier run ===")
    else:
        print(f"=== Evaluating {condition} ===")
        rows, perplexity = evaluate_condition(condition, experiment_config, models_config, chain_lengths)
        for row in rows:
            row["seed"] = SEED
        ps.stages_dir().mkdir(parents=True, exist_ok=True)
        cache_file.write_text(json.dumps({"rows": rows, "perplexity": perplexity}), encoding="utf-8")
        ps.finish(stage)
    all_rows.extend(rows)
    perplexities[condition] = perplexity
    print(f"{condition}: {len(rows)} tasks, perplexity={perplexity:.2f}")
    output = write_eval_outputs(all_rows, perplexities, REPO_ROOT / "results")

print("wrote results/eval_results.jsonl, eval_results.csv, eval_summary.json")

=== Evaluating base ===
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.7: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
total weights      : 3.257 GiB
no_split classes   : ['Qwen3DecoderLayer']
output head        : lm_head -> cuda:1
head headroom      : 0.359 GiB
activation reserve : 11.166 GiB requested
tied to head       : ['model.embed_tokens']
  cuda:0  budget  12.95 GiB  weights  1.760 GiB  free 11.192 GiB  reserve 11.166 GiB
  cuda:1  budget  13.00 GiB  weights  1.497 GiB  free 11.499 GiB  reserve

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Unsloth 2026.9.7 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.
`use_return_dict` is deprecated! Use `return_dict` instead!


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.


[persist] saved to NahlaNabil/adbench-run (stage eval_distilled)
distilled: 221 tasks, perplexity=18.42
wrote results/eval_results.jsonl, eval_results.csv, eval_summary.json


In [9]:
import pandas as pd

df = pd.DataFrame(all_rows)[["condition", "task_set", "chain_length", "success"]]
table = df.groupby(["task_set", "condition", "chain_length"])["success"].mean().unstack("chain_length").round(3)
print(f"seed {SEED}: full-chain success")
table

seed 0: full-chain success


chain_length                   1      3      5
task_set     condition                        
seen_tools   base          1.000    NaN    NaN
             distilled     1.000    NaN    NaN
             self_distill  1.000    NaN    NaN
             sft_early     1.000    NaN    NaN
             sft_ls        1.000    NaN    NaN
             sft_only      1.000    NaN    NaN
unseen_tools base          0.913  0.725  0.743
             distilled     1.000  0.900  0.914
             self_distill  0.891  0.875  0.800
             sft_early     1.000  0.850  0.714
             sft_ls        0.957  0.525  0.457
             sft_only      0.957  0.400  0.229

In [10]:
ps.push(f"final results, seed {SEED}")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.


[persist] saved to NahlaNabil/adbench-run (final results, seed 0)


True